<a href="https://colab.research.google.com/github/ramcharan170/Automated_lead_scoring_agent/blob/mohithra-work/crisp_dm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn sentence-transformers transformers accelerate torch

In [ ]:
import pandas as pd
import numpy as np

# 1. Load Data Dictionary
print("--- DATA DICTIONARY DEFINITIONS ---")
dict_df = pd.read_excel('Leads Data Dictionary.xlsx')
pd.set_option('display.max_colwidth', None)
display(dict_df)

# 2. Load Raw Dataset
df = pd.read_csv('Lead Scoring.csv')
print(f"\nDataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")

# 3. Check Target Conversion Rate
print("--- TARGET VARIABLE DISTRIBUTION ('Converted') ---")
print(df['Converted'].value_counts(normalize=True) * 100)

# 4. Detect Unselected Dropdown Placeholders ('Select')
select_counts = (df == 'Select').sum()
print("\n--- COLUMNS CONTAINING 'Select' PLACEHOLDERS ---")
print(select_counts[select_counts > 0])

--- DATA DICTIONARY DEFINITIONS ---


,Unnamed: 0,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,NaN,Variables,Description
2,NaN,Prospect ID,A unique ID with which the customer is identified.
3,NaN,Lead Number,A lead number assigned to each lead procured.
4,NaN,Lead Origin,"The origin identifier with which the customer was identified to be a lead. Includes API, Landing Page Submission, etc."
5,NaN,Lead Source,"The source of the lead. Includes Google, Organic Search, Olark Chat, etc."
6,NaN,Do Not Email,An indicator variable selected by the customer wherein they select whether of not they want to be emailed about the course or not.
7,NaN,Do Not Call,An indicator variable selected by the customer wherein they select whether of not they want to be called about the course or not.
8,NaN,Converted,The target variable. Indicates whether a lead has been successfully converted or not.
9,NaN,TotalVisits,The total number of visits made by the customer on the website.



Dataset Dimensions: 9240 rows, 37 columns

--- TARGET VARIABLE DISTRIBUTION ('Converted') ---
Converted
0    61.461039
1    38.538961
Name: proportion, dtype: float64

--- COLUMNS CONTAINING 'Select' PLACEHOLDERS ---
Specialization                        1942
How did you hear about X Education    5043
Lead Profile                          4146
City                                  2249
dtype: int64


In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Replace 'Select' placeholders with NaN
df_clean = df_clean.replace('Select', np.nan)

# Key categorical and numerical columns for profiling
cat_cols = [
    'Lead Source', 'Last Activity', 'Specialization',
    'What is your current occupation',
    'What matters most to you in choosing a course',
    'City'
]
num_cols = ['TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit'] # Corrected column name

# Fill missing categorical values with 'Unknown'
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

# Fill missing numerical values with 0
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(0)

# Build narrative string for Hugging Face vector embeddings
def create_context_string(row):
    return (
        f"Occupation: {row['What is your current occupation']} | "
        f"Specialization: {row['Specialization']} | "
        f"Primary Goal: {row['What matters most to you in choosing a course']} | "
        f"Lead Source: {row['Lead Source']} | "
        f"Last Activity: {row['Last Activity']} | "
        f"City: {row['City']} | "
        f"Time Spent on Site: {int(row['Total Time Spent on Website'])} seconds | "
        f"Total Visits: {int(row['TotalVisits'])}" # Corrected column name
    )

df_clean['Lead_Context_String'] = df_clean.apply(create_context_string, axis=1)

print("Data Cleaning Complete!")
print("\nSample Generated Context String:")
print(df_clean['Lead_Context_String'].iloc[0])

Data Cleaning Complete!

Sample Generated Context String:
Occupation: Unemployed | Specialization: Unknown | Primary Goal: Better Career Prospects | Lead Source: Olark Chat | Last Activity: Page Visited on Website | City: Unknown | Time Spent on Site: 0 seconds | Total Visits: 0


In [ ]:
import pandas as pd
import numpy as np

# 1. Deduplicate based on Prospect ID
df_clean = df.drop_duplicates(subset=['Prospect ID']).copy()

# 2. Hard Filter: Remove leads who opted out of email outreach
df_clean = df_clean[df_clean['Do Not Email'] != 'Yes']

# 3. Replace 'Select' placeholders with NaN
df_clean = df_clean.replace('Select', np.nan)

# 4. Standardize Categorical Channels
df_clean['Lead Source'] = df_clean['Lead Source'].replace({
    'google': 'Google',
    'facebook': 'Facebook',
    'blog': 'Organic Search'
})

# Group rare Lead Sources (appearing less than 10 times) into 'Other'
source_counts = df_clean['Lead Source'].value_counts()
rare_sources = source_counts[source_counts < 10].index
df_clean['Lead Source'] = df_clean['Lead Source'].replace(rare_sources, 'Other')

# 5. Fill missing values
cat_cols = [
    'Lead Source', 'Last Activity', 'Specialization',
    'What is your current occupation',
    'What matters most to you in choosing a course',
    'City'
]
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['TotalVisits'] = df_clean['TotalVisits'].fillna(0)
df_clean['Total Time Spent on Website'] = df_clean['Total Time Spent on Website'].fillna(0)

# Cap extreme outliers in TotalVisits at the 99th percentile
cap_visits = df_clean['TotalVisits'].quantile(0.99)
df_clean['TotalVisits'] = np.where(df_clean['TotalVisits'] > cap_visits, cap_visits, df_clean['TotalVisits'])

# 6. Feature Engineering: Convert raw time spent (seconds) to qualitative engagement bands
def convert_time_to_band(seconds):
    if seconds == 0:
        return "No Site Engagement"
    elif seconds < 180:
        return f"Low Engagement ({int(seconds // 60)} mins)"
    elif seconds < 600:
        return f"Moderate Engagement ({int(seconds // 60)} mins)"
    else:
        return f"High Engagement ({int(seconds // 60)} mins)"

df_clean['Engagement_Band'] = df_clean['Total Time Spent on Website'].apply(convert_time_to_band)


In [ ]:
# ----------------------------------------------------
# MODIFICATION 1: DOMAIN DENSITY & CONTEXT STRING ENRICHMENT
# ----------------------------------------------------
# 1. Compute historical conversion counts per domain
# ====================================================
# MODIFICATION 1: DOMAIN TIERING & NARRATIVE STRING
# ====================================================
converted_df_all = df_clean[df_clean['Converted'] == 1]
domain_stats = converted_df_all.groupby('Specialization').size().to_dict()

def get_domain_tier(specialization):
    conversions = domain_stats.get(specialization, 0)
    if conversions > 20:
        return f"High-Converting Sector ({conversions} past buyers)"
    elif conversions > 5:
        return f"Active Sector ({conversions} past buyers)"
    else:
        return f"Emerging Sector ({conversions} past buyers)"

df_clean['Domain_Tier'] = df_clean['Specialization'].apply(get_domain_tier)

def build_domain_context_string(row):
    return (
        f"Domain Sector: {row['Specialization']} [{row['Domain_Tier']}] | "
        f"Role: {row['What is your current occupation']} | "
        f"Goal: {row['What matters most to you in choosing a course']} | "
        f"Channel: {row['Lead Source']} | "
        f"City: {row['City']} | "
        f"Engagement: {row['Engagement_Band']}"
    )

df_clean['Lead_Context_String'] = df_clean.apply(build_domain_context_string, axis=1)

# Sample 100 benchmark buyers & 50 target prospects
converted_df = df_clean[df_clean['Converted'] == 1].sample(n=100, random_state=42).copy()
unscored_df = df_clean[df_clean['Converted'] == 0].sample(n=50, random_state=42).copy()

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load lightweight Hugging Face embedding model
print("Loading SentenceTransformer model ('all-MiniLM-L6-v2')...")
# ====================================================
# MODIFICATION 2: EMBEDDINGS & DOMAIN AFFINITY BOOST
# ====================================================
embedder = SentenceTransformer('all-MiniLM-L6-v2')

converted_vectors = embedder.encode(converted_df['Lead_Context_String'].tolist(), show_progress_bar=True)
unscored_vectors = embedder.encode(unscored_df['Lead_Context_String'].tolist(), show_progress_bar=True)

sim_matrix = cosine_similarity(unscored_vectors, converted_vectors)
lead_scores = []
similar_customers_formatted = []
for i, (target_idx, target_row) in enumerate(unscored_df.iterrows()):
    target_domain = target_row['Specialization']
    target_sims = sim_matrix[i].copy()
    # Apply 1.20x boost for matching domain
    for j, (conv_idx, conv_row) in enumerate(converted_df.iterrows()):
        if conv_row['Specialization'] == target_domain and target_domain != 'Unknown':
            target_sims[j] = min(1.0, target_sims[j] * 1.20)
    top_2_indices = np.argsort(target_sims)[-2:][::-1]
    best_score = round(float(target_sims[top_2_indices[0]]) * 100, 2)
    lead_scores.append(best_score)
    matches_text = []
    for rank, match_idx in enumerate(top_2_indices, 1):
        match_row = converted_df.iloc[match_idx]
        match_sim = round(float(target_sims[match_idx]) * 100, 1)
        matches_text.append(
            f"{rank}. {match_row['Specialization']} ({match_row['What is your current occupation']}) - "
            f"Converted via {match_row['Lead Source']} with {match_row['Engagement_Band']} (Match: {match_sim}%)."
        )
    similar_customers_formatted.append("\n".join(matches_text))

Loading SentenceTransformer model ('all-MiniLM-L6-v2')...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from google.colab import files
# Build DataFrame with the exact 8 columns requested by your partner
# ====================================================
# MODIFICATION 3: EXPORT 50 PROSPECTS TO CSV
# ====================================================
scored_leads_df = pd.DataFrame({
    'lead_id': "LEAD_" + unscored_df['Lead Number'].astype(str),
    'lead_name': "Prospect " + unscored_df['Lead Number'].astype(str),
    'lead_title': unscored_df['What is your current occupation'],
    'company_name': "Unknown",
    'industry': unscored_df['Specialization'],
    'company_size': 100,
    'lead_score': lead_scores,
    'similar_customers': similar_customers_formatted
}).sort_values(by='lead_score', ascending=False)
output_filename = 'scored_leads.csv'
scored_leads_df.to_csv(output_filename, index=False)
files.download(output_filename)
print(f"SUCCESS: Exported {len(scored_leads_df)} scored leads to '{output_filename}'!")
display(scored_leads_df.head(5))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

SUCCESS: Exported 50 scored leads to 'scored_leads.csv'!


,lead_id,lead_name,lead_title,company_name,industry,company_size,lead_score,similar_customers
5216,LEAD_609819,Prospect 609819,Unemployed,Unknown,IT Projects Management,100,100.0,1. IT Projects Management (Unemployed) - Converted via Google with High Engagement (33 mins) (Match: 100.0%).\n2. IT Projects Management (Unknown) - Converted via Google with High Engagement (18 mins) (Match: 100.0%).
5810,LEAD_605254,Prospect 605254,Unemployed,Unknown,Marketing Management,100,100.0,1. Marketing Management (Unemployed) - Converted via Google with Moderate Engagement (8 mins) (Match: 100.0%).\n2. Marketing Management (Unemployed) - Converted via Direct Traffic with High Engagement (24 mins) (Match: 100.0%).
3490,LEAD_626996,Prospect 626996,Unemployed,Unknown,Finance Management,100,100.0,1. Finance Management (Working Professional) - Converted via Google with Moderate Engagement (9 mins) (Match: 100.0%).\n2. Finance Management (Working Professional) - Converted via Reference with No Site Engagement (Match: 100.0%).
7924,LEAD_589021,Prospect 589021,Unknown,Unknown,Unknown,100,100.0,1. Unknown (Unknown) - Converted via Olark Chat with No Site Engagement (Match: 100.0%).\n2. Unknown (Unknown) - Converted via Olark Chat with No Site Engagement (Match: 100.0%).
5920,LEAD_604412,Prospect 604412,Unemployed,Unknown,E-COMMERCE,100,100.0,1. E-COMMERCE (Working Professional) - Converted via Direct Traffic with Low Engagement (1 mins) (Match: 100.0%).\n2. Marketing Management (Unemployed) - Converted via Google with High Engagement (13 mins) (Match: 95.0%).
